# Simulacion de Trafico
## Carriles Exclusivos para Motocicletas Electricas

Este notebook simula y compara dos escenarios de trafico:
1. **Trafico mixto**: carros y motos comparten carriles
2. **Carril exclusivo**: motos electricas tienen su propio carril

---
**Instrucciones**: Ejecuta cada celda en orden (Shift + Enter)

In [ ]:
# Instalacion de dependencias (ya vienen en Colab)
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation
import numpy as np
from dataclasses import dataclass
from typing import List, Optional, Tuple
from IPython.display import HTML, display

print("Dependencias cargadas correctamente!")

## Configuracion
Puedes modificar estos valores para experimentar con diferentes escenarios:

In [ ]:
CONFIG = {
    # VEHICULOS
    "num_carros": 25,
    "num_motos": 18,

    # VELOCIDADES
    "velocidad_carro_min": 1.5,
    "velocidad_carro_max": 3.5,
    "velocidad_moto_min": 5.0,
    "velocidad_moto_max": 7.5,

    # VIA
    "longitud_via": 500,

    # SIMULACION
    "pasos_simulacion": 200,
    "num_simulaciones": 10,

    # CONGESTION
    "distancia_seguridad": 35,
    "factor_congestion_mixto": 0.25,

    # SOSTENIBILIDAD
    "emision_moto_gasolina": 72,  # g CO2/km
}

print(f"Configuracion:")
print(f"   Carros: {CONFIG['num_carros']}")
print(f"   Motos:  {CONFIG['num_motos']}")
print(f"   Longitud via: {CONFIG['longitud_via']} unidades")

## Clases y Funciones

In [ ]:
@dataclass
class Vehiculo:
    """Representa un vehiculo en la simulacion."""
    id: int
    tipo: str
    posicion: float
    velocidad_base: float
    carril: int = 0
    tiempo_inicio: int = 0
    tiempo_llegada: Optional[int] = None
    distancia_recorrida: float = 0
    color: str = ""

    def __post_init__(self):
        self.velocidad_actual = self.velocidad_base
        if not self.color:
            if self.tipo == "carro":
                self.color = random.choice(['#3498db', '#2c3e50', '#7f8c8d', '#c0392b', '#8e44ad'])
            else:
                self.color = '#27ae60'

    def mover(self, velocidad_efectiva: float):
        self.posicion += velocidad_efectiva
        self.distancia_recorrida += velocidad_efectiva

    def ha_llegado(self, longitud_via: float) -> bool:
        return self.posicion >= longitud_via


def crear_vehiculo(id: int, tipo: str, config: dict, carril: int = 0) -> Vehiculo:
    if tipo == "carro":
        velocidad = random.uniform(config["velocidad_carro_min"], config["velocidad_carro_max"])
    else:
        velocidad = random.uniform(config["velocidad_moto_min"], config["velocidad_moto_max"])
    posicion_inicial = random.uniform(0, config["longitud_via"] * 0.3)
    return Vehiculo(id=id, tipo=tipo, posicion=posicion_inicial, velocidad_base=velocidad, carril=carril)


print("Clases definidas correctamente!")

In [ ]:
class SimulacionBasica:
    """Simulacion que compara trafico mixto vs carril exclusivo."""

    def __init__(self, config: dict):
        self.config = config
        self.vehiculos: List[Vehiculo] = []
        self.paso_actual = 0

    def inicializar_vehiculos(self):
        self.vehiculos = []
        for i in range(self.config["num_carros"]):
            self.vehiculos.append(crear_vehiculo(i, "carro", self.config))
        for i in range(self.config["num_motos"]):
            self.vehiculos.append(crear_vehiculo(i + self.config["num_carros"], "moto", self.config))

    def calcular_congestion(self, vehiculo: Vehiculo, es_mixto: bool) -> float:
        dist_seguridad = self.config["distancia_seguridad"]
        vehiculos_adelante = [
            v for v in self.vehiculos
            if v.id != vehiculo.id and v.tiempo_llegada is None
            and v.posicion > vehiculo.posicion
            and v.posicion - vehiculo.posicion < dist_seguridad * 2
        ]

        if not es_mixto:
            if vehiculo.tipo == "moto":
                return max(0.9, 1.0 - len(vehiculos_adelante) * 0.02)
            carros_adelante = [v for v in vehiculos_adelante if v.tipo == "carro"]
            return max(0.6, 1.0 - len(carros_adelante) * 0.1) if carros_adelante else 1.0

        if not vehiculos_adelante:
            return 1.0

        carros = sum(1 for v in vehiculos_adelante if v.tipo == "carro")
        motos = sum(1 for v in vehiculos_adelante if v.tipo == "moto")

        if vehiculo.tipo == "moto":
            factor = 1.0 - (carros * 0.15) - (motos * 0.05)
            factor *= (1 - self.config["factor_congestion_mixto"] * 0.3)
            return max(0.35, factor)
        return max(0.45, 1.0 - (carros * 0.12) - (motos * 0.03))

    def ejecutar_paso(self, es_mixto: bool):
        self.paso_actual += 1
        for v in self.vehiculos:
            if v.tiempo_llegada is not None:
                continue
            factor = self.calcular_congestion(v, es_mixto)
            v.mover(v.velocidad_base * factor)
            if v.ha_llegado(self.config["longitud_via"]):
                v.tiempo_llegada = self.paso_actual

    def ejecutar_simulacion(self, es_mixto: bool) -> dict:
        self.inicializar_vehiculos()
        self.paso_actual = 0
        for _ in range(self.config["pasos_simulacion"]):
            self.ejecutar_paso(es_mixto)
        return self.calcular_metricas()

    def calcular_metricas(self) -> dict:
        tiempos_carros, tiempos_motos = [], []
        for v in self.vehiculos:
            if v.tiempo_llegada is not None:
                t = v.tiempo_llegada - v.tiempo_inicio
                (tiempos_carros if v.tipo == "carro" else tiempos_motos).append(t)

        t_carros = sum(tiempos_carros) / len(tiempos_carros) if tiempos_carros else float('inf')
        t_motos = sum(tiempos_motos) / len(tiempos_motos) if tiempos_motos else float('inf')
        total = len(self.vehiculos)
        llegaron = len(tiempos_carros) + len(tiempos_motos)

        return {
            "tiempo_promedio_carros": t_carros,
            "tiempo_promedio_motos": t_motos,
            "carros_llegaron": len(tiempos_carros),
            "motos_llegaron": len(tiempos_motos),
            "eficiencia": (llegaron / total) * 100 if total > 0 else 0,
        }

print("Simulacion basica definida!")

---
## 1. Ejecutar Simulacion y Ver Graficas
Ejecuta la siguiente celda para ver los resultados comparativos:

In [ ]:
def ejecutar_simulaciones(config):
    resultados_mixto = {"tiempo_promedio_carros": [], "tiempo_promedio_motos": [], "eficiencia": []}
    resultados_exclusivo = {"tiempo_promedio_carros": [], "tiempo_promedio_motos": [], "eficiencia": []}

    print(f"Ejecutando {config['num_simulaciones']} simulaciones...")

    for i in range(config["num_simulaciones"]):
        sim = SimulacionBasica(config)

        r_mixto = sim.ejecutar_simulacion(es_mixto=True)
        for k in resultados_mixto:
            resultados_mixto[k].append(r_mixto[k])

        r_excl = sim.ejecutar_simulacion(es_mixto=False)
        for k in resultados_exclusivo:
            resultados_exclusivo[k].append(r_excl[k])

        print(f"  Simulacion {i+1}/{config['num_simulaciones']} completada")

    def prom(lista):
        v = [x for x in lista if x != float('inf')]
        return sum(v) / len(v) if v else 0

    return {k: prom(v) for k, v in resultados_mixto.items()}, {k: prom(v) for k, v in resultados_exclusivo.items()}

# Ejecutar
mixto, exclusivo = ejecutar_simulaciones(CONFIG)

# Mostrar resultados
print("\n" + "="*50)
print("RESULTADOS")
print("="*50)
print(f"\nTRAFICO MIXTO:")
print(f"   Tiempo carros: {mixto['tiempo_promedio_carros']:.1f}")
print(f"   Tiempo motos:  {mixto['tiempo_promedio_motos']:.1f}")
print(f"   Eficiencia:    {mixto['eficiencia']:.1f}%")
print(f"\nCARRIL EXCLUSIVO:")
print(f"   Tiempo carros: {exclusivo['tiempo_promedio_carros']:.1f}")
print(f"   Tiempo motos:  {exclusivo['tiempo_promedio_motos']:.1f}")
print(f"   Eficiencia:    {exclusivo['eficiencia']:.1f}%")

if mixto['tiempo_promedio_motos'] > 0:
    mejora = (mixto['tiempo_promedio_motos'] - exclusivo['tiempo_promedio_motos']) / mixto['tiempo_promedio_motos'] * 100
    print(f"\n>>> MEJORA PARA MOTOS: {mejora:.1f}% menos tiempo <<<")

In [ ]:
# GRAFICAS COMPARATIVAS
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Comparacion: Trafico Mixto vs Carril Exclusivo', fontsize=14, fontweight='bold')

colores = ['#e74c3c', '#27ae60']
labels = ['Mixto', 'Exclusivo']

# Grafica 1: Tiempos por tipo
ax1 = axes[0]
tiempos = [mixto["tiempo_promedio_carros"], exclusivo["tiempo_promedio_carros"],
           mixto["tiempo_promedio_motos"], exclusivo["tiempo_promedio_motos"]]
bars = ax1.bar([0, 1, 2, 3], tiempos, color=[colores[0], colores[1], colores[0], colores[1]], edgecolor='black')
ax1.set_xticks([0, 1, 2, 3])
ax1.set_xticklabels(['Carros\nMixto', 'Carros\nExcl', 'Motos\nMixto', 'Motos\nExcl'])
ax1.set_ylabel('Tiempo promedio (pasos)')
ax1.set_title('Tiempos de Viaje')
for bar, t in zip(bars, tiempos):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{t:.0f}', ha='center', fontweight='bold')

# Grafica 2: Motos especificamente
ax2 = axes[1]
bars2 = ax2.bar(labels, [mixto["tiempo_promedio_motos"], exclusivo["tiempo_promedio_motos"]], color=colores, edgecolor='black')
ax2.set_ylabel('Tiempo promedio (pasos)')
ax2.set_title('Tiempo de Viaje - MOTOS')
if mixto["tiempo_promedio_motos"] > 0:
    mejora = (mixto["tiempo_promedio_motos"] - exclusivo["tiempo_promedio_motos"]) / mixto["tiempo_promedio_motos"] * 100
    ax2.annotate(f'{mejora:.0f}% menos\ntiempo', xy=(1, exclusivo["tiempo_promedio_motos"]),
                xytext=(0.5, (mixto["tiempo_promedio_motos"] + exclusivo["tiempo_promedio_motos"])/2),
                fontsize=12, color='green', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='green', lw=2))

# Grafica 3: Eficiencia
ax3 = axes[2]
bars3 = ax3.bar(labels, [mixto["eficiencia"], exclusivo["eficiencia"]], color=colores, edgecolor='black')
ax3.set_ylabel('Eficiencia (%)')
ax3.set_title('Vehiculos que Llegaron')
ax3.set_ylim(0, 110)
ax3.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
for bar, e in zip(bars3, [mixto["eficiencia"], exclusivo["eficiencia"]]):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{e:.0f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 2. Ver Animacion
Ejecuta la siguiente celda para ver la animacion de la simulacion:

In [ ]:
class SimulacionVisual:
    def __init__(self, config):
        self.config = config
        self.paso_actual = 0
        self.vehiculos_mixto = []
        self.vehiculos_exclusivo = []
        self.llegados_mixto = {"carros": 0, "motos": 0}
        self.llegados_exclusivo = {"carros": 0, "motos": 0}

    def inicializar(self):
        random.seed(42)
        self.vehiculos_mixto = []
        self.vehiculos_exclusivo = []

        for i in range(self.config["num_carros"]):
            v_m = crear_vehiculo(i, "carro", self.config, random.choice([0, 1]))
            v_e = crear_vehiculo(i, "carro", self.config, random.choice([0, 1]))
            v_e.posicion = v_m.posicion
            self.vehiculos_mixto.append(v_m)
            self.vehiculos_exclusivo.append(v_e)

        for i in range(self.config["num_motos"]):
            id_m = i + self.config["num_carros"]
            v_m = crear_vehiculo(id_m, "moto", self.config, random.choice([0, 1]))
            v_e = crear_vehiculo(id_m, "moto", self.config, 2)
            v_e.posicion = v_m.posicion
            self.vehiculos_mixto.append(v_m)
            self.vehiculos_exclusivo.append(v_e)

    def calcular_congestion(self, v, vehiculos, es_excl):
        dist = self.config["distancia_seguridad"]
        adelante = [x for x in vehiculos if x.id != v.id and x.carril == v.carril
                   and x.tiempo_llegada is None and x.posicion > v.posicion]
        dist_min = min([x.posicion - v.posicion for x in adelante], default=float('inf'))

        if es_excl and v.tipo == "moto":
            return 0.95 if dist_min > dist else max(0.6, dist_min / dist)
        if not es_excl and v.tipo == "moto":
            for x in vehiculos:
                if x.tipo == "carro" and x.tiempo_llegada is None and x.posicion > v.posicion:
                    if x.posicion - v.posicion < dist * 2:
                        return self.config["factor_congestion_mixto"]
        return 1.0 if dist_min > dist * 3 else max(0.15, dist_min / dist * 0.5)

    def mover(self, vehiculos, es_excl):
        for v in vehiculos:
            if v.tiempo_llegada:
                continue
            f = self.calcular_congestion(v, vehiculos, es_excl)
            v.mover(v.velocidad_base * f)
            if v.posicion >= self.config["longitud_via"]:
                v.tiempo_llegada = self.paso_actual

    def contar(self):
        self.llegados_mixto = {"carros": 0, "motos": 0}
        self.llegados_exclusivo = {"carros": 0, "motos": 0}
        for v in self.vehiculos_mixto:
            if v.tiempo_llegada:
                self.llegados_mixto["carros" if v.tipo == "carro" else "motos"] += 1
        for v in self.vehiculos_exclusivo:
            if v.tiempo_llegada:
                self.llegados_exclusivo["carros" if v.tipo == "carro" else "motos"] += 1

    def paso(self):
        self.paso_actual += 1
        self.mover(self.vehiculos_mixto, False)
        self.mover(self.vehiculos_exclusivo, True)
        self.contar()

print("Simulacion visual definida!")

In [ ]:
# CREAR ANIMACION
sim = SimulacionVisual(CONFIG)
sim.inicializar()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('Simulacion de Trafico en Tiempo Real', fontsize=14, fontweight='bold', color='white')

def dibujar_via(ax, vehiculos, es_excl, titulo, llegados):
    ax.clear()
    ax.set_facecolor('#16213e')
    L = CONFIG["longitud_via"]
    ax.set_xlim(-10, L + 50)
    ax.set_ylim(-2, 25)
    ax.axis('off')

    color_titulo = '#27ae60' if es_excl else '#e74c3c'
    ax.text(L/2, 23, titulo, ha='center', fontsize=12, fontweight='bold', color=color_titulo)

    # Carriles
    for i in range(2):
        ax.add_patch(patches.Rectangle((0, i*7), L, 6, facecolor='#2c3e50', edgecolor='white', lw=1))
    if es_excl:
        ax.add_patch(patches.Rectangle((0, 14), L, 5, facecolor='#0a4d1c', edgecolor='#00ff00', lw=2))
        ax.text(L/2, 16.5, 'CARRIL EXCLUSIVO MOTOS', ha='center', color='#00ff00', fontsize=9)

    # Meta
    ax.add_patch(patches.Rectangle((L-8, 0), 8, 19 if es_excl else 14, facecolor='#27ae60', alpha=0.8))
    ax.text(L-4, 9, 'META', ha='center', va='center', color='white', fontweight='bold')

    # Vehiculos
    for v in vehiculos:
        if v.tiempo_llegada:
            continue
        y = v.carril * 7 + 3 if v.carril < 2 else 16.5
        if v.tipo == "carro":
            ax.add_patch(patches.FancyBboxPatch((v.posicion-8, y-2), 16, 4,
                        boxstyle="round,pad=0.2", facecolor=v.color, edgecolor='black', lw=1))
        else:
            color = '#00ff00' if es_excl else '#ff6b35'
            ax.add_patch(patches.Circle((v.posicion, y), 2.5, facecolor=color, edgecolor='black', lw=1))

    # Contador
    ax.text(10, -1, f"Carros: {llegados['carros']}/{CONFIG['num_carros']}  |  Motos: {llegados['motos']}/{CONFIG['num_motos']}",
           fontsize=10, color='white')

def actualizar(frame):
    sim.paso()
    dibujar_via(ax1, sim.vehiculos_mixto, False, 'SIN CARRIL EXCLUSIVO - Motos bloqueadas', sim.llegados_mixto)
    dibujar_via(ax2, sim.vehiculos_exclusivo, True, 'CON CARRIL EXCLUSIVO - Motos fluyen libres', sim.llegados_exclusivo)
    return []

# Crear animacion
anim = animation.FuncAnimation(fig, actualizar, frames=150, interval=80, blit=False)

# Mostrar en Colab
plt.close()
HTML(anim.to_jshtml())

---
## 3. Metricas de Sostenibilidad
Impacto ambiental de usar motos electricas:

In [ ]:
# Calcular metricas de sostenibilidad
km_motos = sum(v.distancia_recorrida / 1000 for v in sim.vehiculos_exclusivo if v.tipo == "moto")
co2_evitado = km_motos * CONFIG["emision_moto_gasolina"] / 1000
arboles = co2_evitado / 22 * 365

print("="*50)
print("IMPACTO AMBIENTAL - MOTOS ELECTRICAS")
print("="*50)
print(f"\n   Kilometros recorridos (electrico): {km_motos:.1f} km")
print(f"   CO2 evitado: {co2_evitado:.3f} kg")
print(f"   Equivalente a: {arboles:.1f} arboles plantados/año")
print(f"\n   Emision moto gasolina: {CONFIG['emision_moto_gasolina']} g CO2/km")
print(f"   Emision moto electrica: 0 g CO2/km")
print("\n>>> Las motos electricas + carril exclusivo =")
print("    movilidad mas rapida Y sostenible <<<")

---
## Conclusiones

Esta simulacion demuestra que:

1. **Las motos se benefician significativamente** de tener un carril exclusivo
2. **Los carros no se perjudican** (incluso pueden mejorar ligeramente)
3. **La eficiencia general del trafico aumenta**
4. **Las motos electricas reducen emisiones** a cero en comparacion con las de gasolina

---
*Proyecto universitario - Simulacion de Trafico*